# Sesión 10 · Cuantiles

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

## 1 · Los datos

In [ ]:
library(tidyverse)

url <- paste0("https://docs.google.com/spreadsheets/d/e/2PACX-1vTTKC57eWZVIQ9lwhoR5nV",
              "qYM4kgi5zA9yifa-YStdfdJwNe7ATs0p-TUCUwvjfcWmHmvsZDEK8VX4I/pub?gid=1326008067&single=true&output=csv")

grupos <- read_csv(url, show_col_types = FALSE) |>
  select(
    carrera  = Carrera,
    genero   = `Género`,
    estatura = `Estatura, en metros`,
    traslado = `Tiempo de traslado a la universidad, en minutos`
  ) |>
  filter(!is.na(traslado), !is.na(estatura))

glimpse(grupos)

## 2 · Primero a mano

Estos son los tiempos de once compañeros, ordenados de menor a mayor.

Antes de calcular: ¿qué valor parte esta lista en dos mitades?

In [ ]:
once <- sort(head(grupos$traslado, 11))
once

Un cuartil se localiza primero por su **posición** en la lista ordenada:

$$Q_k = \frac{k\,(n+1)}{4}$$

Con 11 datos, el primer cuartil está en la posición $\frac{1 \times 12}{4} = 3$,
o sea el tercer dato.

In [ ]:
n <- length(once)

(n + 1) * 0.25      # la posición
once[3]             # el dato que está en esa posición

In [ ]:
# Los tres cortes de una vez
posiciones <- (n + 1) * c(0.25, 0.50, 0.75)
posiciones

once[posiciones]

In [ ]:
# Y ahora la función de R
quantile(once, probs = c(0.25, 0.50, 0.75))

### Dos fórmulas para la misma posición

La mediana y el tercer cuartil coinciden. **El primero no:** a mano dio 8 y
`quantile()` devolvió 11.5.

Cada método ubica el cuartil en un lugar distinto de la lista:

| | Fórmula de la posición | Con $n = 11$ y $p = 0{,}25$ |
|---|---|---|
| A mano, **tipo 6** | $(n+1)\,p$ | $12 \times 0{,}25 = 3$ |
| `quantile()`, **tipo 7** | $1 + (n-1)\,p$ | $1 + 10 \times 0{,}25 = 3{,}5$ |

La posición 3 es un dato de la lista: el **8**.

La posición 3,5 cae entre el tercero y el cuarto, así que se **interpola**:

$$8 + 0{,}5 \times (15 - 8) = 11{,}5$$

Con el argumento `type` se le puede pedir a R la misma fórmula del cálculo a
mano.

In [ ]:
quantile(once, probs = c(0.25, 0.50, 0.75), type = 6)   # la fórmula de arriba

### Por qué hay más de una

Un cuantil de una muestra no tiene definición única. Las nueve que trae R
difieren en dónde colocan la posición y en cómo reparten el peso entre los dos
datos vecinos.

In [ ]:
x <- c(12, 14, 15, 18, 19, 22, 25, 28, 31, 40)
n <- length(x)
p <- 0.25

tibble(
  tipo     = c(4, 5, 6, 7, 8),
  formula  = c("np", "np + 0.5", "(n+1)p", "1 + (n-1)p", "(n+1/3)p + 1/3"),
  posicion = c(n*p, n*p + 0.5, (n+1)*p, 1 + (n-1)*p, (n + 1/3)*p + 1/3),
  Q1       = sapply(c(4, 5, 6, 7, 8), function(t) quantile(x, p, type = t))
)

**Ninguna de las nueve es la correcta: son convenciones.** Lo que no se vale es
mezclarlas dentro del mismo trabajo, ni reportar un cuartil sin saber de cuál
salió.

En este curso se usa `quantile()` tal cual, que es el tipo 7.

> Hyndman, R. J. y Fan, Y. (1996). Sample Quantiles in Statistical Packages.
> *The American Statistician*, 50(4), 361-365. Es la referencia que cita la
> ayuda de `quantile()`.

En Excel pasa lo mismo: `PERCENTIL.INC` es el tipo 7 y `PERCENTIL.EXC` es el
tipo 6. Si comparas una cifra tuya contra una calculada en Excel o contra un
libro que use $(n+1)p$, los números van a diferir y la diferencia no es un error.

En Excel pasa lo mismo: `PERCENTIL.INC` es el tipo 7 y `PERCENTIL.EXC` es el
tipo 6.

Si le preguntas a un modelo de lenguaje cuál es el primer cuartil, vas a recibir
un número sin saber con qué convención se calculó. La pregunta útil es cuál de
las nueve definiciones usó, y si es la misma que usa el resto de tu trabajo.

## 3 · Cuartiles, mediana y rango intercuartil

In [ ]:
quantile(grupos$traslado, probs = c(0.25, 0.50, 0.75))

El segundo cuartil es la **mediana**: parte al grupo en dos mitades del mismo
tamaño.

$$Q_2 = D_5 = P_{50}$$

El **rango intercuartil** es la distancia entre el primer y el tercer cuartil, o
sea el tramo donde vive la mitad central:

$$RIC = Q_3 - Q_1$$

In [ ]:
q <- quantile(grupos$traslado, c(0.25, 0.75))
q[2] - q[1]

IQR(grupos$traslado)

## 4 · Deciles

Nueve cortes. `seq()` los genera sin escribirlos uno por uno.

In [ ]:
seq(0.1, 0.9, by = 0.1)

In [ ]:
quantile(grupos$traslado, probs = seq(0.1, 0.9, by = 0.1))

### Tu apuesta

Piensa en el ingreso mensual de tu hogar, sumando lo que entra de todas las
personas que viven ahí. **¿En cuál de los diez deciles de México crees que cae?**

Contesta en el formulario que se proyecta en clase. Es anónimo: no pide nombre ni
correo, y aquí solo aparece el conteo del grupo.

### El conteo del grupo

La celda de abajo lee las respuestas y arma la tabla. `.drop = FALSE` conserva
los deciles donde nadie se ubicó: un cero también es información.

In [ ]:
url_decil <- "PEGA_AQUI_LA_URL_DEL_CSV"

NIVELES <- c("I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X")

conteo <- read_csv(url_decil, show_col_types = FALSE) |>
  rename(decil = 2) |>                        # la 1 es la marca de tiempo
  mutate(
    decil = str_extract(decil, "^[IVX]+"),     # "I · el de menor..." -> "I"
    decil = factor(decil, levels = NIVELES)    # niveles explícitos y en orden
  ) |>
  count(decil, .drop = FALSE) |>               # conserva los deciles en cero
  mutate(`% del grupo` = round(100 * n / sum(n), 1))

conteo

In [ ]:
ggplot(conteo, aes(x = decil, y = `% del grupo`)) +
  geom_col(fill = "#3A6B6F") +
  geom_hline(yintercept = 10, color = "#A4503C", linetype = "dashed", linewidth = 1) +
  annotate("text", x = 9.2, y = 11.5, label = "10% nacional", color = "#A4503C") +
  labs(x = "Decil de ingreso", y = "% del grupo")

Por definición, cada decil contiene el **10%** de los hogares del país: la línea
punteada. Si el grupo se apila en el centro, la diferencia contra esa línea es lo
que hay que explicar.

Alrededor del 40% de las personas en países de la OCDE se ubica en el centro de
la distribución, coincida o no con su ingreso real. Quienes están arriba
subestiman su posición y quienes están abajo la sobreestiman; entre los hogares
del 30% más bajo, el 70% se coloca más arriba de donde está. Y subestiman más
quienes tienen más años de escolaridad.

### Los deciles reales

In [ ]:
deciles <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/enigh2024_deciles.csv", show_col_types = FALSE)

deciles |>
  transmute(
    decil,
    ingreso_mensual = round(mensual),
    hasta           = round(tope_mensual),
    porcentaje_del_ingreso = round(porcentaje_del_ingreso, 1)
  )

El decil dice en qué lugar del reparto cae un hogar. No dice si ese ingreso
alcanza: eso depende de cuántas personas viven de él, de dónde viven y de qué
cuestan ahí las cosas.

El decil X incluye tanto al hogar que recibe 50 mil pesos al mes como al que
recibe millones. Un corte por posición no describe lo que pasa dentro del último
tramo.

## 5 · Percentiles

Cien partes, para cuando las diez de los deciles resultan gruesas.

In [ ]:
quantile(grupos$traslado, probs = c(0.05, 0.90, 0.95, 0.99))

Con este número de respuestas, el percentil 37 y el 38 casi siempre caen entre
los dos mismos datos. El resultado sale con decimales y eso no significa que esté
medido con esa precisión.

## 6 · El resumen de cinco números

In [ ]:
quantile(grupos$traslado, probs = seq(0, 1, by = 0.25))

Mínimo, $Q_1$, mediana, $Q_3$ y máximo describen la distribución completa con
cinco valores. Son los que dibuja el diagrama de caja de la próxima sesión.

## 7 · Tu percentil

Cambia el número por tu propio tiempo de traslado.

In [ ]:
mi_traslado <- 30

mean(grupos$traslado <= mi_traslado) * 100

## Tarea

Sobre **tu serie del portafolio**:

1. Los cuartiles de tu variable numérica, y una oración que interprete $Q_1$ y
   $Q_3$ con las unidades de tu serie.
2. El rango intercuartil, y qué tramo de tus datos describe.
3. Un percentil que sirva para decidir algo en tu dominio, elegido por ti, con la
   razón de por qué ese y no otro.
4. Una línea sobre qué no dice ese percentil.

**Guarda tu copia.**